# 24. XGBoost — `xgboost`
**Python Libraries Lab · Phase 2 · Machine Learning**

_Gradient-boosted trees: the go-to model for tabular data._

✅ **Runs in Colab** as-is.

XGBoost builds hundreds of small decision trees, one after another. Each new tree focuses on the mistakes the previous trees made. Added together, these weak trees become a very strong model. This idea is called gradient boosting, and for tables of numbers and categories it often beats every other method.

**In this notebook:** ShopEasy wants to forecast how much each order will be worth. You will train an XGBoost regressor, compare it with a simple baseline, use early stopping, and look at which features drive the prediction.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q xgboost
import importlib.metadata as md
for pkg in ['xgboost']:
    print(pkg, md.version(pkg))

## Step 1 · Make the data
Predict `order_value` from customer and basket facts.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(0)
n = 5000
df = pd.DataFrame({
    "items": rng.integers(1, 10, n),
    "avg_item_price": rng.choice([199, 499, 999, 1499, 2999], n),
    "is_member": rng.integers(0, 2, n),
    "festival_week": (rng.random(n) < 0.15).astype(int),
    "hour": rng.integers(0, 24, n),
    "city_code": rng.integers(0, 4, n),
})
df["order_value"] = (df["items"] * df["avg_item_price"] * (1 + 0.25 * df["festival_week"]) * (0.9 + 0.1 * df["is_member"])
                     + 300 * np.sin(df["hour"] / 24 * 2 * np.pi) + rng.normal(0, 400, n)).round()
X, y = df.drop(columns="order_value"), df["order_value"]
df.head()

## Step 2 · Train, validation and test sets
Train to learn, validation for early stopping, test for the final honest grade.

In [ ]:
from sklearn.model_selection import train_test_split
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.2, random_state=0)
print(len(X_train), len(X_val), len(X_test))

## Step 3 · Baseline first
Always know what 'no model' scores. Here: predict the average for everyone.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
baseline = np.full(len(y_test), y_train.mean())
print(f"Baseline MAE: Rs.{mean_absolute_error(y_test, baseline):,.0f}")

## Step 4 · Train XGBoost with early stopping
It stops adding trees when the validation error has not improved for 50 rounds.

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.05, max_depth=5,
                         subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=50, random_state=0)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
pred = model.predict(X_test)
print("Trees used:", model.best_iteration + 1)
print(f"XGBoost MAE:  Rs.{mean_absolute_error(y_test, pred):,.0f}")
print(f"XGBoost RMSE: Rs.{root_mean_squared_error(y_test, pred):,.0f}")

## Step 5 · What drives the prediction?
Feature importance by 'gain': how much each feature improved the trees.

In [ ]:
import matplotlib.pyplot as plt
xgb.plot_importance(model, importance_type="gain", show_values=False, title="Feature importance (gain)")
plt.show()

example = pd.DataFrame([{"items": 3, "avg_item_price": 999, "is_member": 1, "festival_week": 1, "hour": 20, "city_code": 2}])
print(f"Predicted value for a festival-week basket: Rs.{model.predict(example)[0]:,.0f}")

## ✍️ Your turn
Train `xgb.XGBRegressor(n_estimators=200, max_depth=3, random_state=0)` on `X_train, y_train` (no early stopping) and store its **test MAE** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    m = xgb.XGBRegressor(n_estimators=200, max_depth=3, random_state=0).fit(X_train, y_train)
    exp = mean_absolute_error(y_test, m.predict(X_test))
    ok = answer is not None and abs(float(answer) - exp) < 1e-6
    return ok, f"Expected about {exp:,.1f}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
m = xgb.XGBRegressor(n_estimators=200, max_depth=3, random_state=0).fit(X_train, y_train)
answer = mean_absolute_error(y_test, m.predict(X_test))

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Boosting**
- **n_estimators**
- **learning_rate**
- **max_depth**
- **Early stopping**
- **RMSE**

**Next:** 25. LightGBM & CatBoost (`lightgbm · catboost`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/25_lightgbm_catboost.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install xgboost
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.